In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp

In [0]:
class BronzeIngestor:
    """
    Author: Gustavo Lourenço
    Description: Classe para ingestão de dados no formato bronze da tabela SalesLT.Customer do SQL Server
    """
    def __init__(
        self,
        spark: SparkSession,
        secret_scope: str,
        catalog: str,
        user_secret_key: str = "sql-user",
        password_secret_key: str = "sql-password",
    ):
        self.spark = spark
        self.secret_scope = secret_scope
        self.catalog = catalog
        self.user_secret_key = user_secret_key
        self.password_secret_key = password_secret_key
        self.dbutils = self._get_dbutils()
        self.urlserver = "svlabgus.database.windows.net"
        
    def _get_dbutils(self):
        """
        Description: Método para obter o dbutils do spark utilizando a função get_ipython().user_ns.get("dbutils")
        Input: None
        Output: dbutils
        """
        try:
            import IPython
            dbutils = IPython.get_ipython().user_ns.get("dbutils")
            if dbutils is not None:
                return dbutils
        except Exception:
            pass

        from pyspark.dbutils import DBUtils
        return DBUtils(self.spark)

    def _get_secret(self, primary_key: str, *fallback_keys: str) -> str:
        keys_to_try = [primary_key, *fallback_keys]
        last_error = None

        for key in keys_to_try:
            try:
                return self.dbutils.secrets.get(scope=self.secret_scope, key=key)
            except Exception as exc:
                last_error = exc

        tried_keys = ", ".join(keys_to_try)
        raise ValueError(
            f"Nenhum secret foi encontrado no scope '{self.secret_scope}'. Chaves testadas: {tried_keys}"
        ) from last_error

    def execute(self, db_server: str, db_name: str, table_origem: str, table_destino: str):
        """
        Description: Método para realizar a ingestão de dados no formato bronze da tabela SalesLT.Customer do SQL Server
        Input: db_server, db_name, table_origem, table_destino
        Output: None
        """
        jdbc_url = f"jdbc:sqlserver://{db_server}:1433;database={db_name}"
        user = self._get_secret(self.user_secret_key, "sql_user", "user")
        password = self._get_secret(self.password_secret_key, "sql_password", "password")
        
        import time

        last_error = None
        for attempt in range(1, 7):
            try:
                df = self.spark.read.format("jdbc")\
                    .option("url", jdbc_url)\
                    .option("user", user)\
                    .option("password", password)\
                    .option("dbtable", table_origem)\
                    .load()
                break
            except Exception as exc:
                last_error = exc
                error_message = str(exc)
                is_transient_pause = "not currently available" in error_message.lower()

                if not is_transient_pause or attempt == 6:
                    raise

                print(f"Banco indisponível no momento. Tentativa {attempt}/6. Aguardando 20 segundos para novo teste...")
                time.sleep(20)
            
        if last_error is not None and 'df' not in locals():
            raise last_error
            
        self.spark.sql(f"CREATE SCHEMA IF NOT EXISTS {self.catalog}.bronze")
        full_dest = f"{self.catalog}.bronze.{table_destino}"
        df.withColumn("_ingestion_time", current_timestamp())\
          .write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(full_dest)
        
        self.spark.sql(f"OPTIMIZE {full_dest}")

if __name__ == "__main__":
    spark = SparkSession.builder.appName("Ingestao_Bronze").getOrCreate()
    ingestor = BronzeIngestor(
        spark,
        "kvault-adventureworks",
        "db_lab_brq",
        user_secret_key="sql-user",
        password_secret_key="sql-password",
    )
    ingestor.execute(ingestor.urlserver, "db_adventureworks_lt", "SalesLT.Customer", "adventureworks_customers")